In [24]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DateType
)

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 26, Finished, Available, Finished, False)

In [25]:
# Source location
landing_path = "Files/Landing/customers.csv"

# Explicit source schema
customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("customer_type", StringType(), True),
    StructField("credit_terms_days", IntegerType(), True),
    StructField("primary_freight_type", StringType(), True),
    StructField("account_status", StringType(), True),
    StructField("contract_start_date", DateType(), True),
    StructField("annual_revenue_potential", IntegerType(), True)
])

# Read Landing data
df_source = (
    spark.read
    .option("header", "true")
    .schema(customer_schema)
    .csv(landing_path)
)

display(df_source)

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a265f21e-c801-4ef4-906b-9661cb4d1942)

In [26]:
# Record count
source_count = df_source.count()

# NULL primary key check
null_customer_ids = (
    df_source
    .filter(F.col("customer_id").isNull())
    .count()
)

# Duplicate primary key check
duplicate_customer_ids = (
    df_source
    .groupBy("customer_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# Business-rule checks
negative_credit_terms = (
    df_source
    .filter(F.col("credit_terms_days") < 0)
    .count()
)

negative_revenue = (
    df_source
    .filter(F.col("annual_revenue_potential") < 0)
    .count()
)

print(f"Source records: {source_count}")
print(f"NULL customer IDs: {null_customer_ids}")
print(f"Duplicate customer IDs: {duplicate_customer_ids}")
print(f"Negative credit terms: {negative_credit_terms}")
print(f"Negative revenue values: {negative_revenue}")

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 28, Finished, Available, Finished, False)

Source records: 200
NULL customer IDs: 0
Duplicate customer IDs: 0
Negative credit terms: 0
Negative revenue values: 0


In [27]:
if null_customer_ids > 0:
    raise ValueError("ETL failed: NULL customer_id values detected.")

if duplicate_customer_ids > 0:
    raise ValueError("ETL failed: Duplicate customer_id values detected.")

if negative_credit_terms > 0:
    raise ValueError("ETL failed: Negative credit terms detected.")

if negative_revenue > 0:
    raise ValueError("ETL failed: Negative annual revenue detected.")

print("Data quality validation passed.")

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 29, Finished, Available, Finished, False)

Data quality validation passed.


In [28]:
df_source = (
    df_source
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file", F.lit("customers.csv"))
)

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 30, Finished, Available, Finished, False)

In [29]:
df_source.createOrReplaceTempView("customers_source")

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 31, Finished, Available, Finished, False)

In [38]:
bronze_customers_path = "Tables/dbo/bronze_customers"

try:
    spark.read.format("delta").load(bronze_customers_path).createOrReplaceTempView("t_bronze_customers")
    print("bronze_customers already exists — skipping creation.")

except Exception:
    v_create_table = """
    CREATE TABLE IF NOT EXISTS bronze_customers (
        customer_id STRING,
        customer_name STRING,
        customer_type STRING,
        credit_terms_days INT,
        primary_freight_type STRING,
        account_status STRING,
        contract_start_date DATE,
        annual_revenue_potential INT,
        ingestion_timestamp TIMESTAMP,
        source_file STRING
    )
    """
    spark.sql(v_create_table)
    print("bronze_customers created.")

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 40, Finished, Available, Finished, False)

bronze_customers created.


In [39]:
spark.sql("""
MERGE INTO bronze_customers AS target
USING customers_source AS source

ON target.customer_id = source.customer_id

WHEN MATCHED THEN
    UPDATE SET
        target.customer_name = source.customer_name,
        target.customer_type = source.customer_type,
        target.credit_terms_days = source.credit_terms_days,
        target.primary_freight_type = source.primary_freight_type,
        target.account_status = source.account_status,
        target.contract_start_date = source.contract_start_date,
        target.annual_revenue_potential = source.annual_revenue_potential,
        target.ingestion_timestamp = source.ingestion_timestamp,
        target.source_file = source.source_file

WHEN NOT MATCHED THEN
    INSERT (
        customer_id,
        customer_name,
        customer_type,
        credit_terms_days,
        primary_freight_type,
        account_status,
        contract_start_date,
        annual_revenue_potential,
        ingestion_timestamp,
        source_file
    )
    VALUES (
        source.customer_id,
        source.customer_name,
        source.customer_type,
        source.credit_terms_days,
        source.primary_freight_type,
        source.account_status,
        source.contract_start_date,
        source.annual_revenue_potential,
        source.ingestion_timestamp,
        source.source_file
    )
""")

StatementMeta(, 0e5211d5-8a3e-472a-b90e-79c677013c95, 41, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]